In [1]:
import sys
import os
import MetaTrader5 as mt5
import datetime
import time

# Import required classes
import MetaTrader5 as mt5

project_dir = os.path.abspath("..")
if project_dir not in sys.path:
    sys.path.append(project_dir)

import polars as pl

from src.infra.mtBase import mtBase
from src.infra.OrderData import OrderData
from src.infra.OrderClient import OrderClient

In [2]:
mtb = mtBase(
    account="mt5demo_acc_usd",
    credentials_path=os.path.join("..", "secrets", "mt5_acc_cred.yaml"),
    config_path=os.path.join("..", "secrets", "mt5_config.ini"),
)
mtb.mt5_init()

MetaTrader 5 connection established


In [3]:
def get_spread_rel(symbol: str) -> float:
    pricetick = mtb.get_symbol_price(symbol, wait_sec=0.2)
    ask_price = pricetick['ask']
    bid_price = pricetick['bid']
    spread = ask_price - bid_price
    spread_rel = spread / (bid_price + 1e-6)
    return spread_rel

In [4]:
start_dt = datetime.datetime.now()
print(start_dt)
duration = datetime.timedelta(minutes=40)
end_dt = start_dt + duration
print(end_dt)

2025-11-01 13:47:17.860957
2025-11-01 14:27:17.860957


In [ ]:
sym_list = ["KO", "INTC", "ALGN", "MU", "SRPT", "VFC"]

# end_dt must be defined, e.g.:
# end_dt = dt.datetime.now() + dt.timedelta(minutes=30)

res: dict[str, list[float]] = {sym: [] for sym in sym_list}

# align to the next minute boundary
next_tick = (datetime.datetime.now().replace(second=0, microsecond=0)
             + datetime.timedelta(minutes=1))

while datetime.datetime.now() < end_dt:
    # sleep until the start of the next minute
    sleep_s = (next_tick - datetime.datetime.now()).total_seconds()
    if sleep_s > 0:
        time.sleep(sleep_s)

    ts = datetime.datetime.now()
    for symbol in sym_list:
        try:
            spread_rel = get_spread_rel(symbol)  # returns a fraction (e.g., 0.0012)
            print(f"{ts:%Y-%m-%d %H:%M:%S} - {symbol}: Spread relative: {spread_rel*100:.4f}%")
            res[symbol].append(spread_rel)
        except Exception as e:
            print(f"{ts:%Y-%m-%d %H:%M:%S} - {symbol}: ERROR {e}")
            res[symbol].append(float("nan"))

    next_tick += datetime.timedelta(minutes=1)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Create plots showing relative spreads for each symbol
def plot_spreads(res):
    """Plot relative spreads for each symbol"""
    
    # Convert results to DataFrame for easier plotting
    df_data = []
    for symbol, spreads in res.items():
        for i, spread in enumerate(spreads):
            if not np.isnan(spread):
                df_data.append({
                    'symbol': symbol,
                    'minute': i + 1,
                    'spread_pct': spread * 100  # Convert to percentage
                })
    
    df = pd.DataFrame(df_data)
    
    if df.empty:
        print("No data to plot. Make sure to run the data collection loop first.")
        return
    
    # Create subplots - one for each symbol
    symbols = list(res.keys())
    n_symbols = len(symbols)
    
    fig, axes = plt.subplots(n_symbols, 1, figsize=(12, 3 * n_symbols))
    if n_symbols == 1:
        axes = [axes]
    
    for i, symbol in enumerate(symbols):
        symbol_data = df[df['symbol'] == symbol]
        if not symbol_data.empty:
            axes[i].plot(symbol_data['minute'], symbol_data['spread_pct'], 
                        marker='o', linewidth=2, markersize=4)
            axes[i].set_title(f'{symbol} - Relative Spread Over Time')
            axes[i].set_xlabel('Minute')
            axes[i].set_ylabel('Spread (%)')
            axes[i].grid(True, alpha=0.3)
            axes[i].set_ylim(bottom=0)
    
    plt.tight_layout()
    plt.show()
    
    # Also create a combined plot
    plt.figure(figsize=(12, 6))
    for symbol in symbols:
        symbol_data = df[df['symbol'] == symbol]
        if not symbol_data.empty:
            plt.plot(symbol_data['minute'], symbol_data['spread_pct'], 
                    marker='o', label=symbol, linewidth=2, markersize=4)
    
    plt.title('Relative Spreads Comparison - All Symbols')
    plt.xlabel('Minute')
    plt.ylabel('Spread (%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(bottom=0)
    plt.show()
    
    # Print summary statistics
    print("\nSpread Summary Statistics:")
    print("-" * 50)
    for symbol in symbols:
        spreads = [s for s in res[symbol] if not np.isnan(s)]
        if spreads:
            spreads_pct = [s * 100 for s in spreads]
            print(f"{symbol:>6}: Mean={np.mean(spreads_pct):.4f}%, "
                  f"Std={np.std(spreads_pct):.4f}%, "
                  f"Min={np.min(spreads_pct):.4f}%, "
                  f"Max={np.max(spreads_pct):.4f}%")
        else:
            print(f"{symbol:>6}: No valid data")

# Check if res variable exists and plot
try:
    plot_spreads(res)
except NameError:
    print("Data not collected yet. Please run the data collection loop (cell 5) first to populate the 'res' variable.")

KeyboardInterrupt: 